In [1]:
import os, time, random, math
from dataclasses import dataclass
from typing import List, Dict, Tuple, Set

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, QED
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


In [3]:
DATA_PATH = "./cache_qm9/qm9_splits_vocab.pt"  
obj = torch.load(DATA_PATH)

In [4]:
train_sm, val_sm, test_sm = obj["splits"]
stoi, itos = obj["stoi"], obj["itos"]
cfg0 = obj.get("cfg", {})

PAD, BOS, EOS = "<PAD>", "<BOS>", "<EOS>"
pad_id = stoi[PAD]
bos_id = stoi[BOS]
eos_id = stoi[EOS]

print("Loaded splits:", len(train_sm), len(val_sm), len(test_sm))
print("Vocab size:", len(itos))

Loaded splits: 120496 6694 6695
Vocab size: 24


In [5]:
def encode(sm: str, stoi: Dict[str,int], max_len: int) -> List[int]:
    ids = [stoi[BOS]] + [stoi[c] for c in sm] + [stoi[EOS]]
    if len(ids) < max_len:
        ids = ids + [stoi[PAD]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
        ids[-1] = stoi[EOS]
    return ids

def decode(ids: List[int], itos: List[str]) -> str:
    stoi_local = {c:i for i,c in enumerate(itos)}
    out = []
    for i in ids:
        if i == stoi_local[EOS]:
            break
        if i in (stoi_local[PAD], stoi_local[BOS]):
            continue
        out.append(itos[i])
    return "".join(out)

MAX_LEN = 80


In [6]:
class SmilesLMDataset(Dataset):
    def __init__(self, smiles_list: List[str], stoi: Dict[str,int], max_len: int):
        self.smiles = smiles_list
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        ids = encode(self.smiles[idx], self.stoi, self.max_len)
        x = torch.tensor(ids[:-1], dtype=torch.long)  # (T-1)
        y = torch.tensor(ids[1:], dtype=torch.long)   # (T-1)
        return x, y

train_ds = SmilesLMDataset(train_sm, stoi, MAX_LEN)
val_ds   = SmilesLMDataset(val_sm, stoi, MAX_LEN)

BATCH_SIZE = 256 if DEVICE=="cuda" else 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(DEVICE=="cuda"))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(DEVICE=="cuda"))

print("Train batches:", len(train_loader), "Val batches:", len(val_loader))


Train batches: 471 Val batches: 27


In [7]:
@dataclass
class LMConfig:
    vocab_size: int
    emb_dim: int = 256
    hidden_dim: int = 512
    num_layers: int = 2
    dropout: float = 0.1

class GRULM(nn.Module):
    def __init__(self, cfg: LMConfig, pad_id: int):
        super().__init__()
        self.cfg = cfg
        self.pad_id = pad_id
        self.embed = nn.Embedding(cfg.vocab_size, cfg.emb_dim, padding_idx=pad_id)
        self.gru = nn.GRU(
            input_size=cfg.emb_dim,
            hidden_size=cfg.hidden_dim,
            num_layers=cfg.num_layers,
            batch_first=True,
            dropout=cfg.dropout if cfg.num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(cfg.hidden_dim, cfg.vocab_size)

    def forward(self, x, h0=None):
        # x: (B, T)
        emb = self.embed(x)          # (B, T, E)
        out, h = self.gru(emb, h0)   # out: (B, T, H)
        logits = self.fc(out)        # (B, T, V)
        return logits, h

lm_cfg = LMConfig(vocab_size=len(itos))
model = GRULM(lm_cfg, pad_id).to(DEVICE)
print(model)


GRULM(
  (embed): Embedding(24, 256, padding_idx=0)
  (gru): GRU(256, 512, num_layers=2, batch_first=True, dropout=0.1)
  (fc): Linear(in_features=512, out_features=24, bias=True)
)


In [8]:
def compute_loss(logits, targets, pad_id: int):
    # logits: (B, T, V), targets: (B, T)
    B, T, V = logits.shape
    loss = F.cross_entropy(
        logits.view(B*T, V),
        targets.view(B*T),
        ignore_index=pad_id
    )
    return loss

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = 0.0
    n = 0
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        logits, _ = model(x)
        loss = compute_loss(logits, y, pad_id)

        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()
        n += 1
    return total_loss / max(1, n)

EPOCHS = 3  # enough for mid-report baseline; increase later
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    t1 = time.time()
    print(f"Epoch {epoch:02d} | train loss {tr:.4f} | val loss {va:.4f} | {t1-t0:.1f}s")


Epoch 01 | train loss 0.9413 | val loss 0.8305 | 13.2s
Epoch 02 | train loss 0.8206 | val loss 0.8094 | 12.5s
Epoch 03 | train loss 0.8066 | val loss 0.8044 | 12.5s


In [9]:
@torch.no_grad()
def sample_lm(model: GRULM, n: int, max_len: int, temperature: float = 1.0) -> List[str]:
    model.eval()
    samples = []
    for _ in range(n):
        x = torch.tensor([[bos_id]], device=DEVICE)  # start with BOS
        h = None
        out_ids = []
        for t in range(max_len - 1):
            logits, h = model(x, h)           # logits: (1, 1, V)
            logits = logits[:, -1, :] / max(1e-8, temperature)
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)  # (1,1)
            nid = int(next_id.item())
            if nid == eos_id:
                break
            if nid != pad_id and nid != bos_id:
                out_ids.append(nid)
            x = next_id  # feed next token
        sm = "".join(itos[i] for i in out_ids)
        samples.append(sm)
    return samples

# quick smoke test
samples = sample_lm(model, n=10, max_len=MAX_LEN, temperature=0.9)
samples[:5]


['CC1OCC(=O)C1C#N',
 'C#CC1(C(O)C#N)CC1',
 'CCC1=CCCCCO1',
 'CC1(O)C2CC3C1CN32',
 'CC12C=CC3CC1C2O3']

In [10]:
def to_mol(smiles: str):
    return Chem.MolFromSmiles(smiles)

def canonical(smiles: str) -> str | None:
    m = to_mol(smiles)
    if m is None:
        return None
    return Chem.MolToSmiles(m)

# canonical train set for novelty
train_canon: Set[str] = set()
for s in train_sm:
    cs = canonical(s)
    if cs is not None:
        train_canon.add(cs)

def validity(smiles_list: List[str]) -> Tuple[float, List[str]]:
    valid_canon = []
    for s in smiles_list:
        cs = canonical(s)
        if cs is not None:
            valid_canon.append(cs)
    return len(valid_canon) / max(1, len(smiles_list)), valid_canon

def uniqueness(valid_canon: List[str]) -> float:
    return 0.0 if len(valid_canon)==0 else len(set(valid_canon))/len(valid_canon)

def novelty(valid_canon: List[str], train_set: Set[str]) -> float:
    return 0.0 if len(valid_canon)==0 else sum(1 for s in valid_canon if s not in train_set)/len(valid_canon)

def compute_properties(valid_canon: List[str]) -> Dict[str,float]:
    if len(valid_canon)==0:
        return {"mw_mean": math.nan, "logp_mean": math.nan, "qed_mean": math.nan}
    mws, logps, qeds = [], [], []
    for s in valid_canon:
        m = to_mol(s)
        if m is None:
            continue
        mws.append(Descriptors.MolWt(m))
        logps.append(Crippen.MolLogP(m))
        qeds.append(QED.qed(m))
    if len(mws)==0:
        return {"mw_mean": math.nan, "logp_mean": math.nan, "qed_mean": math.nan}
    return {"mw_mean": sum(mws)/len(mws), "logp_mean": sum(logps)/len(logps), "qed_mean": sum(qeds)/len(qeds)}

def evaluate_smiles(samples: List[str], train_set: Set[str]) -> Dict[str,float]:
    v, valid_canon = validity(samples)
    out = {
        "n_samples": len(samples),
        "validity": v,
        "n_valid": len(valid_canon),
        "uniqueness": uniqueness(valid_canon),
        "novelty": novelty(valid_canon, train_set),
    }
    out.update(compute_properties(valid_canon))
    return out


In [11]:
N_SAMPLES = 5000

t0 = time.time()
gen = sample_lm(model, n=N_SAMPLES, max_len=MAX_LEN, temperature=0.9)
t1 = time.time()

metrics = evaluate_smiles(gen, train_canon)
gen_seconds = t1 - t0
metrics["gen_seconds"] = gen_seconds
metrics["samples_per_sec"] = N_SAMPLES / gen_seconds
metrics["valid_per_sec"] = metrics["n_valid"] / gen_seconds

metrics


{'n_samples': 5000,
 'validity': 0.9702,
 'n_valid': 4851,
 'uniqueness': 0.9736136878994022,
 'novelty': 0.24613481756338898,
 'mw_mean': 123.25122861265677,
 'logp_mean': 0.37338908266336823,
 'qed_mean': 0.4641287246680476,
 'gen_seconds': 31.370733499526978,
 'samples_per_sec': 159.38422351761054,
 'valid_per_sec': 154.63457365678573}

In [13]:
valid_canon = [canonical(s) for s in gen if canonical(s) is not None]
valid_set = set(valid_canon)
novel_count = sum(1 for s in valid_set if s not in train_canon)

print("Valid:", len(valid_canon))
print("Unique valid:", len(valid_set))
print("Novel unique:", novel_count)
print("Novel% (unique):", novel_count / max(1, len(valid_set)))

Valid: 4851
Unique valid: 4723
Novel unique: 1177
Novel% (unique): 0.24920601312724963


In [12]:
os.makedirs("./checkpoints", exist_ok=True)
ckpt_path = "./checkpoints/gru_lm_qm9.pt"

torch.save({
    "model_state": model.state_dict(),
    "lm_cfg": lm_cfg.__dict__,
    "stoi": stoi,
    "itos": itos,
    "max_len": MAX_LEN,
}, ckpt_path)

print("Saved:", ckpt_path)

# Save some samples
with open("./checkpoints/gru_lm_samples.txt", "w") as f:
    for s in gen[:200]:
        f.write(s + "\n")
print("Saved sample text file.")


Saved: ./checkpoints/gru_lm_qm9.pt
Saved sample text file.
